# 05 - Comparative Evaluation & Error Analysis (RQ1-3)

Combine all saved predictions into comparison tables and figures, build a
role confusion matrix, and categorize errors.

Produces `results/metrics.json`, `results/predictions.csv`, `results/error_analysis.csv`, and figures under `results/figures/`.

In [1]:
import sys, os
from pathlib import Path
# Make src/ importable whether run from repo root or notebooks/.
ROOT = Path.cwd()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))
elif (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
print('project root:', ROOT)


project root: /Users/ellam/Documents/Projects/openai-srl-benchmark


In [2]:
import json, glob
import pandas as pd
import data_loader as dl, srl_inference as si, evaluation as ev, visualization as viz
test = dl.read_jsonl('data/test.jsonl')

## Load whatever prediction files exist

In [3]:
files = {
    'zero-shot': 'results/predictions_zero_shot.jsonl',
    '2-shot': 'results/predictions_few_shot_2.jsonl',
    '5-shot': 'results/predictions_few_shot_5.jsonl',
    '10-shot': 'results/predictions_few_shot_10.jsonl',
    'fine-tuned': 'results/predictions_fine_tuned.jsonl',
}
import os
loaded = {name: si.load_predictions(p) for name, p in files.items() if os.path.exists(p)}
print('available configs:', list(loaded))

available configs: ['zero-shot', '2-shot', '5-shot', '10-shot']


## Overall comparison table

In [4]:
metrics = {name: ev.evaluate(res, test) for name, res in loaded.items()}
rows = [ev.summary_row(name, m) for name, m in metrics.items()]
summary_df = pd.DataFrame(rows)
summary_df

,configuration,precision,recall,f1,macro_f1,relaxed_f1,predicate_echo_acc,schema_valid_rate,n
0,zero-shot,0.3434,0.3241,0.3335,0.0779,0.3413,0.996,1.0,500
1,2-shot,0.4860,0.4122,0.4461,0.0931,0.4516,1.000,1.0,500
2,5-shot,0.4599,0.4146,0.4361,0.0997,0.4409,1.000,1.0,500
3,10-shot,0.4893,0.4353,0.4607,0.1142,0.4663,1.000,1.0,500


## Per-role F1 table

In [5]:
role_f1 = {name: {r: pr['f1'] for r, pr in m['strict']['per_role'].items()}
           for name, m in metrics.items()}
role_df = pd.DataFrame(role_f1).fillna(0).round(3)
role_df

,zero-shot,2-shot,5-shot,10-shot
ARG0,0.496,0.617,0.621,0.652
ARG1,0.406,0.539,0.551,0.594
ARG2,0.059,0.281,0.254,0.242
ARG3,0.000,0.000,0.000,0.056
ARG4,0.000,0.000,0.000,0.182
ARG5,0.000,0.000,0.000,0.000
ARGM-ADJ,0.000,0.000,0.000,0.000
ARGM-ADV,0.000,0.000,0.000,0.000
ARGM-CAU,0.069,0.000,0.080,0.043
ARGM-COM,0.000,0.000,0.000,0.000


## Figures

In [6]:
if rows:
    viz.plot_overall_f1(rows)
if role_f1:
    core = ['ARG0','ARG1','ARG2','ARG3','ARGM-TMP','ARGM-LOC','ARGM-MNR']
    viz.plot_role_f1_comparison(role_f1, roles=[r for r in core if any(r in d for d in role_f1.values())])
# confusion matrix from the best available config
if loaded:
    best = max(metrics, key=lambda k: metrics[k]['strict']['micro']['f1'])
    conf = ev.role_confusion(loaded[best], test)
    viz.plot_confusion_matrix(conf, filename=f'confusion_{best}.png')
    print('confusion matrix for best config:', best)

confusion matrix for best config: 10-shot


## Error taxonomy

In [7]:
error_frames = []
for name, res in loaded.items():
    errs = ev.classify_errors(res, test)
    for e in errs:
        e['config'] = name
    error_frames.extend(errs)
    print(name, ev.error_category_counts(errs))
error_df = pd.DataFrame(error_frames)
error_df.head(20) if len(error_df) else 'no errors / no predictions yet'

zero-shot {'argument_omission': 583, 'argument_hallucination': 557, 'boundary_error': 19, 'role_confusion': 218, 'adjunct_confusion': 36, 'predicate_echo_error': 2}
2-shot {'argument_hallucination': 321, 'argument_omission': 502, 'role_confusion': 215, 'adjunct_confusion': 17, 'boundary_error': 11}
5-shot {'argument_hallucination': 364, 'argument_omission': 479, 'role_confusion': 216, 'adjunct_confusion': 37, 'boundary_error': 10}
10-shot {'argument_hallucination': 329, 'argument_omission': 451, 'role_confusion': 215, 'adjunct_confusion': 37, 'boundary_error': 13}


,example_id,sentence,predicate,category,gold,predicted,config
0,val-6467,Server COPD can lead to hyperinflation of the ...,hyperinflation,argument_omission,ARG1:of the lungs,,zero-shot
1,val-6467,Server COPD can lead to hyperinflation of the ...,hyperinflation,argument_hallucination,,ARG0:Server COPD,zero-shot
2,val-6467,Server COPD can lead to hyperinflation of the ...,hyperinflation,argument_hallucination,,ARG1:the lungs,zero-shot
3,val-4656,All that needed to happen was for us to tally ...,tally up,argument_hallucination,,ARG2:our money back,zero-shot
4,val-2936,We will not use the security forces of our cou...,silence,boundary_error,ARG0[0:2]:We,ARG0[None:None]:we,zero-shot
5,val-2936,We will not use the security forces of our cou...,silence,argument_hallucination,,ARG2:the security forces of our country,zero-shot
6,val-2936,We will not use the security forces of our cou...,silence,argument_hallucination,,ARGM-DIR:to,zero-shot
7,val-6801,"In a statement , Mr. Merksamer described the f...",necessitated,argument_omission,ARG0:our inability to obtain trade credit,,zero-shot
8,val-6801,"In a statement , Mr. Merksamer described the f...",necessitated,argument_omission,ARGM-ADV:combined with a need to ensure that o...,,zero-shot
9,val-6801,"In a statement , Mr. Merksamer described the f...",necessitated,argument_hallucination,,"ARG0:our inability to obtain trade credit , co...",zero-shot


## Persist results

In [8]:
import json
# full metrics
with open('results/metrics.json', 'w') as fh:
    json.dump({name: m for name, m in metrics.items()}, fh, indent=2)
# flat predictions table across configs
pred_rows = []
for name, res in loaded.items():
    for r in res:
        for a in r['prediction']['arguments']:
            pred_rows.append({'config': name, 'example_id': r['example_id'],
                              'sentence': r['sentence'], 'predicate': r['predicate'],
                              'role': a['role'], 'text': a.get('text'),
                              'start': a.get('start'), 'end': a.get('end')})
pd.DataFrame(pred_rows).to_csv('results/predictions.csv', index=False)
if len(error_frames):
    pd.DataFrame(error_frames).to_csv('results/error_analysis.csv', index=False)
summary_df.to_csv('results/summary_table.csv', index=False)
print('saved results/metrics.json, predictions.csv, error_analysis.csv, summary_table.csv')

saved results/metrics.json, predictions.csv, error_analysis.csv, summary_table.csv
